In [2]:
import sys
import logging
from typing import Dict, Any, List, Optional, Set
from dataclasses import dataclass, field
from enum import Enum
import uuid

# 配置日志输出
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("SideEffectRuntime")

# ============================================================================
# 1. 动作契约定义 (ActionContract)
# ============================================================================

class ActionType(str, Enum):
    READ = "READ"
    WRITE_SIDE_EFFECT = "WRITE_SIDE_EFFECT"  # 具有副作用的写操作（如转账、审批）

@dataclass
class ActionContract:
    tool_name: str
    action_type: ActionType
    required_role: str = "EMPLOYEE"
    sensitive_parameters: Set[str] = field(default_factory=set) # 不可擅自修改的参数

# ============================================================================
# 2. 策略网关决策结果与 PolicyGate (默认拒绝安全策略)
# ============================================================================

class PolicyDecision(str, Enum):
    ALLOW = "ALLOW"
    DENY = "DENY"
    REQUIRE_CONFIRMATION = "REQUIRE_CONFIRMATION"

@dataclass
class PolicyResult:
    decision: PolicyDecision
    reason: str

class PolicyGate:
    @staticmethod
    def evaluate(
        user_role: str,
        contract: ActionContract,
        user_intent_goal: Dict[str, Any],
        agent_proposed_action: Dict[str, Any]
    ) -> PolicyResult:
        """
        综合评估：权限校验 + 副作用确认 + 默认拒绝的安全策略
        """
        # 1. 权限判定 (RBAC Check)
        role_hierarchy = {"EMPLOYEE": 1, "MANAGER": 2, "ADMIN": 3}
        user_level = role_hierarchy.get(user_role, 0)
        required_level = role_hierarchy.get(contract.required_role, 1)
        
        if user_level < required_level:
            return PolicyResult(
                decision=PolicyDecision.DENY,
                reason=f"权限不足: 用户角色 [{user_role}] 无法执行需要 [{contract.required_role}] 权限的操作。"
            )

        # 2. 纯只读操作且无越权 -> 直接放行
        if contract.action_type == ActionType.READ:
            return PolicyResult(decision=PolicyDecision.ALLOW, reason="只读操作，权限与意图匹配。")

        # 3. 针对具有副作用的操作 (Write Side Effect) - 默认拒绝 / 严格校验策略
        expected_action = user_intent_goal.get("action")
        proposed_action_name = agent_proposed_action.get("action") or agent_proposed_action.get("tool_name")
        
        # 默认拒绝：如果用户意图动作与 Agent 提出的动作不一致，直接 DENY 或 REQUIRE_CONFIRMATION
        if expected_action != proposed_action_name:
            return PolicyResult(
                decision=PolicyDecision.REQUIRE_CONFIRMATION,
                reason=f"意图不一致拦截: 用户意图为 [{expected_action}]，但 Agent 尝试执行 [{proposed_action_name}]。"
            )

        # 3.1 检查敏感参数是否发生静默篡改 (Goal Mutation Guard)
        for param in contract.sensitive_parameters:
            original_val = user_intent_goal.get(param)
            proposed_val = agent_proposed_action.get(param)
            if original_val is not None and proposed_val is not None:
                if original_val != proposed_val:
                    return PolicyResult(
                        decision=PolicyDecision.REQUIRE_CONFIRMATION,
                        reason=f"参数篡改拦截: 用户授权参数 [{param}: {original_val}] 被 Agent 修改为 [{param}: {proposed_val}]。"
                    )

        return PolicyResult(
            decision=PolicyDecision.ALLOW,
            reason="副作用写操作安全校验通过：权限符合，意图与参数一致。"
        )

# ============================================================================
# 3. 带幂等性保护与状态一致性的后端与 ToolRuntime
# ============================================================================

@dataclass
class ExecutionResult:
    success: bool
    status_code: int
    data: Dict[str, Any]
    message: str

class MockBankingBackend:
    """模拟真实的银行/支付后端，维护交易账本、账户余额及准确的扣款计数器"""
    def __init__(self):
        self.ledger: Dict[str, ExecutionResult] = {}
        self.account_balance = 5000.0
        self.debit_count = 0  # 明确记录实际完成扣款的次数

    def transfer(self, amount: float, idempotency_key: str, simulate_timeout: bool = False) -> ExecutionResult:
        # 【关键幂等防线】如果该 Key 已经处理过，直接返回历史结果，绝不重复扣款！
        if idempotency_key in self.ledger:
            logger.info(f"🛡️ [Idempotency Check] 捕获到重复提交的 Key: {idempotency_key}。直接返回历史账本结果，未重复扣款。")
            return self.ledger[idempotency_key]

        # 余额校验
        if amount > self.account_balance:
            res = ExecutionResult(success=False, status_code=400, data={}, message="DECLINED: INSUFFICIENT_FUNDS")
            return res

        # 模拟网络 Timeout 异常 (异常发生在数据包发出或返回途中，但后端实际已完成扣款并记录账本)
        if simulate_timeout:
            logger.error(f"💥 [Backend Fault] 支付网关发生 Socket Timeout！(但后端实际已扣款成功)")
            
            # 修复核心：必须在抛出超时前，真实更新 account_balance 并自增 debit_count
            self.account_balance -= amount
            self.debit_count += 1
            
            success_result = ExecutionResult(
                success=True,
                status_code=200,
                data={"balance": self.account_balance, "transferred": amount},
                message="SUCCESS_AFTER_TIMEOUT_RECOVERY"
            )
            # 保持账本记录与余额更新的原子一致性
            self.ledger[idempotency_key] = success_result
            raise TimeoutError("Gateway Read Timeout - Response Dropped in Transit")

        # 正常扣款流程
        self.account_balance -= amount
        self.debit_count += 1
        
        res = ExecutionResult(
            success=True,
            status_code=200,
            data={"balance": self.account_balance, "transferred": amount},
            message="SUCCESS"
        )
        self.ledger[idempotency_key] = res
        return res

    def query_status(self, idempotency_key: str) -> Optional[ExecutionResult]:
        """通过幂等键查询下游状态"""
        return self.ledger.get(idempotency_key)


class ToolRuntime:
    def __init__(self, backend: MockBankingBackend):
        self.backend = backend

    def execute_with_side_effect_guard(
        self,
        action_name: str,
        params: Dict[str, Any],
        idempotency_key: str,
        simulate_timeout: bool = False
    ) -> ExecutionResult:
        """
        强制通过策略网关的安全副作用执行器：
        当写操作遭遇 Timeout 时，决不盲目重新调用发送交易，而是通过查询幂等状态恢复。
        """
        logger.info(f"🚀 [ToolRuntime] 执行写操作: {action_name} | IdempotencyKey: {idempotency_key}")
        
        try:
            return self.backend.transfer(
                amount=params.get("amount", 0.0),
                idempotency_key=idempotency_key,
                simulate_timeout=simulate_timeout
            )
        except TimeoutError as te:
            logger.warning(f"⚠️ [Runtime Catch] 捕获到网络超时错误: {te}")
            logger.info("🔄 [Safe Recovery Protocol] 触发容灾补偿：向后端查询该 IdempotencyKey 的实际执行状态，绝不盲目重试扣款！")
            
            recovered_res = self.backend.query_status(idempotency_key)
            if recovered_res:
                logger.info(f"✅ [Recovery Success] 下游账本确认：该笔交易已安全落地，状态码: {recovered_res.status_code}")
                return recovered_res
            else:
                return ExecutionResult(success=False, status_code=504, data={}, message="TIMEOUT_AND_NO_LEDGER_RECORD")


# ============================================================================
# 4. 验证第五个场景及严格断言
# ============================================================================

def run_day36_scenario_5_test():
    print("=" * 70)
    print("🧪 Day 36: Scenario 5 Timeout Recovery & Idempotency Strict Assertions")
    print("=" * 70)

    backend = MockBankingBackend()
    runtime = ToolRuntime(backend)

    idempotency_key = str(uuid.uuid4())
    print(f"生成的唯一幂等键 (Idempotency Key): {idempotency_key}")
    print(f"初始后端账户余额: {backend.account_balance}")
    print(f"初始扣款计数器 (debit_count): {backend.debit_count}")

    # 第一次调用：人为模拟触发 Timeout，但后端已经成功扣款并记录账本
    print("\n>>> 第一次调用 (触发 Timeout 异常，后端完成扣款并记录账本)...")
    try:
        res_first = runtime.execute_with_side_effect_guard(
            action_name="execute_transfer",
            params={"amount": 1000},
            idempotency_key=idempotency_key,
            simulate_timeout=True
        )
        print(f"运行时最终恢复结果: {res_first.message}, 下游返回余额: {res_first.data.get('balance')}")
    except Exception as e:
        print(f"捕获到异常: {e}")

    print(f"\n--- 第一次调用后的状态检查 ---")
    print(f"后端账户余额: {backend.account_balance}")
    print(f"debit_count: {backend.debit_count}")
    print(f"ledger 长度: {len(backend.ledger)}")

    # 第二次调用（模拟由于超时未收到响应而再次发起相同请求，带同样的 Idempotency Key）
    print("\n>>> 第二次调用 (使用相同的 Idempotency Key 再次提交)...")
    res_second = runtime.execute_with_side_effect_guard(
        action_name="execute_transfer",
        params={"amount": 1000},
        idempotency_key=idempotency_key,
        simulate_timeout=False
    )
    print(f"第二次提交返回结果: {res_second.message}, 下游返回余额: {res_second.data.get('balance')}")

    print(f"\n--- 最终状态与严格断言检查 ---")
    print(f"最终账户余额: {backend.account_balance}")
    print(f"最终 ledger 记录数: {len(backend.ledger)}")
    print(f"最终扣款计数器 (debit_count): {backend.debit_count}")

    # 严格断言
    assert backend.account_balance == 4000.0, f"期望余额为 4000.0，实际为 {backend.account_balance}"
    assert len(backend.ledger) == 1, f"期望账本记录数为 1，实际为 {len(backend.ledger)}"
    assert backend.debit_count == 1, f"期望扣款次数为 1，实际为 {backend.debit_count}"
    
    print("\n✅ 所有严格断言全部通过！账本记录与余额更新保持了一致性，且成功实现了防重复扣款。")

if __name__ == "__main__":
    run_day36_scenario_5_test()

2026-10-20 11:08:13,284 - [INFO] - 🚀 [ToolRuntime] 执行写操作: execute_transfer | IdempotencyKey: 42667331-cb97-41cd-b400-14cd4b72fb6f
2026-10-20 11:08:13,285 - [ERROR] - 💥 [Backend Fault] 支付网关发生 Socket Timeout！(但后端实际已扣款成功)
2026-10-20 11:08:13,287 - [WARNING] - ⚠️ [Runtime Catch] 捕获到网络超时错误: Gateway Read Timeout - Response Dropped in Transit
2026-10-20 11:08:13,288 - [INFO] - 🔄 [Safe Recovery Protocol] 触发容灾补偿：向后端查询该 IdempotencyKey 的实际执行状态，绝不盲目重试扣款！
2026-10-20 11:08:13,289 - [INFO] - ✅ [Recovery Success] 下游账本确认：该笔交易已安全落地，状态码: 200
2026-10-20 11:08:13,290 - [INFO] - 🚀 [ToolRuntime] 执行写操作: execute_transfer | IdempotencyKey: 42667331-cb97-41cd-b400-14cd4b72fb6f
2026-10-20 11:08:13,292 - [INFO] - 🛡️ [Idempotency Check] 捕获到重复提交的 Key: 42667331-cb97-41cd-b400-14cd4b72fb6f。直接返回历史账本结果，未重复扣款。


🧪 Day 36: Scenario 5 Timeout Recovery & Idempotency Strict Assertions
生成的唯一幂等键 (Idempotency Key): 42667331-cb97-41cd-b400-14cd4b72fb6f
初始后端账户余额: 5000.0
初始扣款计数器 (debit_count): 0

>>> 第一次调用 (触发 Timeout 异常，后端完成扣款并记录账本)...
运行时最终恢复结果: SUCCESS_AFTER_TIMEOUT_RECOVERY, 下游返回余额: 4000.0

--- 第一次调用后的状态检查 ---
后端账户余额: 4000.0
debit_count: 1
ledger 长度: 1

>>> 第二次调用 (使用相同的 Idempotency Key 再次提交)...
第二次提交返回结果: SUCCESS_AFTER_TIMEOUT_RECOVERY, 下游返回余额: 4000.0

--- 最终状态与严格断言检查 ---
最终账户余额: 4000.0
最终 ledger 记录数: 1
最终扣款计数器 (debit_count): 1

✅ 所有严格断言全部通过！账本记录与余额更新保持了一致性，且成功实现了防重复扣款。
